In [1]:
import duckdb
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz, utils

In [2]:
db = duckdb.connect('patents.db')
db.execute("PRAGMA memory_limit='3GB'")
db.execute("PRAGMA temp_directory='C:/temp/duckdb_spill'")
db.execute("PRAGMA threads=2")

In [3]:
db.close()

In [3]:
db.sql("SHOW TABLES")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ patents_classified      │
│ patents_embeddings      │
│ patents_labelled        │
│ run_260710_0000         │
│ run_260710_0000_reverse │
└─────────────────────────┘

In [4]:
#db.sql("DESCRIBE run_260617_1048")
db.sql("DESCRIBE patents_labelled")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ family_id          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ application_number │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ title              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cpc                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_year   │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ jurisdiction       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_dimensions    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

In [51]:
#df = db.sql("SELECT * FROM publications_embedding").df()
df = db.sql("SELECT * FROM patents_labelled").df()

In [6]:
df

,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,date_dimensions,date_ML,date_LLM,scope_curated,pillar_curated,date_labelled,research_category,end_product,ingredient
0,KR-20260058202-A,87847814,KR1020267005189,Powdered food emulsion composition containing ...,The present invention relates to a powdered fo...,"['A23P10/40', 'A23L2/39', 'A23C11/06', 'A23C11...",2026,KR,260710,260710,260710,in,PB,260710,End product formulation,Cross-cutting,"Emulsions, gels, and binders"
1,WO-2025074118-A1,93119608,GB2024/052562,METHODS FOR INCREASING THE FAT CONTENT OF A CELL,Described herein are novel methods for increas...,"['C12N5/0653', 'C12N2501/01', 'C12N2527/00']",2025,WO,260710,260710,260710,in,CM,260710,Cell line development,Meat,Fats and oils
2,EP-4638706-A2,89542141,EP23838012.5,NANOFIBROUS SCAFFOLD FOR CELL CULTURING,The present invention relates to a method for ...,"['C12N2513/00', 'C12M33/00', 'C12M25/02', 'C12...",2025,EP,260710,260710,260710,in,CM,260710,Scaffolding,Agnostic,N/A
3,PL-247657-B1,91227479,PL442891,Method for producing instant vegetable steaks ...,The subject of the application is an instant v...,"['A23L19/00', 'A23L33/125', 'A23L33/105', 'A23...",2025,PL,260710,260710,260710,in,PB,260710,End product formulation,Meat,N/A
4,US-20250000129-A1,83692816,US18690539,USE OF LEGUMINOUS STARCH AND ITS CROSS LINKED ...,The present invention is relative to the use o...,"['A23L29/219', 'A23L13/426', 'A23J3/227', 'A23...",2025,US,260710,260710,260710,in,PB,260710,End product formulation,Meat,"Emulsions, gels, and binders"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,WO-2022136708-A1,74105752,EP2021/087661,PRODUCTION OF FUNGAL BIOMASS,The present invention relates to a method for ...,"[Y02E50/10, A23L31/00, A23J3/20, C12N1/22, A23...",2022,WO,260710,260710,260710,in,F,260710,Feedstocks,Agnostic,N/A
446,WO-2022103318-A1,78695771,SE2021/051128,A GENETICALLY MODIFIED YEAST CELL FOR HEMOGLOB...,"A genetically modified yeast cell, wherein the...","[C12N2510/00, C07K14/395, C12N15/81, C12R2001/...",2022,WO,260710,260710,260710,in,F,260710,Target molecule selection,Meat,N/A
447,WO-2022058287-A1,74095986,EP2021/075137,"A MICROBIAL CELL PRODUCT, METHOD FOR OBTAINING...",The present invention relates to a method for ...,"[A23J3/225, A23J3/227, A23L33/145, A23L33/135,...",2022,WO,260710,260710,260710,in,F,260710,Ingredient optimisation,Eggs and egg proteins,"Emulsions, gels, and binders"
448,WO-2021104846-A1,73059967,EP2020/081588,POWDERED COMPOSITION COMPRISING A SOLID PLANT-...,The present disclosure relates to a powdered c...,"[A23K20/158, A23D7/05, A23D9/00, A23L29/10, A2...",2021,WO,260710,260710,260710,in,PB,260710,Ingredient optimisation,Meat,Fats and oils


In [6]:
# clean abstract
df['abstract'] = df['abstract'].str.replace(r'<[^>]*>', '', regex=True)

In [40]:
company_list = pd.read_csv("company_list.csv")
company_database = pd.read_csv("company_database.csv")

companies = pd.concat([company_list['Cleaned-up Name'], company_database['Company']]).drop_duplicates().tolist()


In [50]:
companies

['21st.BIO',
 '70/30 Food Tech',
 'Adamo Foods',
 'Aerbio (Deep Branch)',
 'Agro Ludens',
 'ÄIO',
 'Air Protein',
 'Algama',
 'Algenuity',
 'Algrow Biosciences',
 'All G',
 'Allium Bio',
 'Almendra',
 'Alver',
 'Amai Proteins',
 'Angel Yeast',
 'Aqua Cultured Foods',
 'Ark Biotech',
 'Arkeon Biotechnologies',
 'Asahi Group',
 'Asterix Foods',
 'Balletic Foods',
 'Beijing Technology and Business University',
 'Better Dairy',
 'Better Meat Company',
 'Better Nature Ltd',
 'betterland foods',
 'Bevo Biotehnoloske Resitve doo',
 'Biofect Innovations',
 'BK GIULINI GMBH',
 'Blue Canopy',
 'Bolder Foods',
 'Bond Pet Foods',
 'Bosque Foods',
 'Brevel',
 'Brig Bio',
 'C&dac',
 'c16 Biosciences',
 'Calidris Bio',
 'Calysta',
 'Cargill Food Ingredients',
 'Cassius AB',
 'catchfree',
 'Central South University of Forestry and Technology (China)',
 'Change Foods',
 'Changing Bio',
 'Checkerspot',
 'China Meat Research Centre',
 'CHUNG JAE WOOK',
 'Chunk Foods',
 'Circe Biotech',
 'CJ CheilJedang',

In [51]:
nomatch = company_list[~company_list["Cleaned-up Name"].values.isin(company_database['Company'])]
nomatch = nomatch.drop_duplicates(subset="Cleaned-up Name")

In [8]:
df["assignee_names"]

0      [VISHWAKARMA INSTITUTE OF TECHNOLOGY]
1                     [NEW SCHOOL FOODS INC]
2                                [PALEO B V]
3                             [UNIV QINGDAO]
4                           [UNIV SOUTHWEST]
                       ...                  
233                              [Mynu Corp]
234                 [Umami Bioworks Pte Ltd]
235                       [Donaldson Co Inc]
236                            [Senara GmbH]
237                   [PRAIRIE AQUATECH LLC]
Name: assignee_names, Length: 238, dtype: object

In [9]:
name_map = dict(zip(company_list['Name'], company_list['Cleaned-up Name']))

In [ ]:
df['assignee_names_cleaned'] = df['assignee_names'].apply(
    lambda x: name_map.get(x, x) if not isinstance(x, (list, np.ndarray)) 
              else [name_map.get(n, n) for n in x]
)

In [75]:
df['assignee_names_cleaned'] = df['assignee_names_cleaned'].apply(
    lambda names: [n.upper() for n in names] if isinstance(names, list) else names
)

In [76]:
import unicodedata

def normalize(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')

def match_company(names, companies, cutoff=90):
    companies_normalized = [normalize(c) for c in companies]
    
    def get_match(n):
        result = process.extractOne(
            normalize(n), companies_normalized,
            scorer=fuzz.WRatio,
            processor=utils.default_process,
            score_cutoff=cutoff
        )
        if result:
            match, score, idx = result
            original_match = companies[idx]  # get original with accents
            if len(original_match) >= len(n) * 0.5:
                return original_match
        return n
    
    if isinstance(names, (list, np.ndarray)):
        return [get_match(n) for n in names]
    return names

df['assignee_names_matched'] = df['assignee_names_cleaned'].apply(
    lambda x: match_company(x, companies)
)

In [68]:
df['assignee_names_cleaned']

0      [VISHWAKARMA INSTITUTE OF TECHNOLOGY]
1                     [NEW SCHOOL FOODS INC]
2                                    [paleo]
3                             [UNIV QINGDAO]
4                           [UNIV SOUTHWEST]
                       ...                  
233                              [Mynu Corp]
234                 [Umami Bioworks Pte Ltd]
235                       [Donaldson Co Inc]
236                            [Senara GmbH]
237                                 [Houdek]
Name: assignee_names_cleaned, Length: 238, dtype: object

In [80]:
df[df['assignee_names_cleaned'] != df['assignee_names_matched']]

,id,title,abstract,application_number,assignee_cities,assignee_countries,assignee_names,family_count,family_id,filing_status,...,times_cited,year,cpc,current_assignee_names,publications,proba_scope,pred_combined,pred_pillar,assignee_names_cleaned,assignee_names_matched
1,ZA-202311713-B,"PROCESS FOR PRODUCING COOKABLE, FIBROUS MEAT A...",The present disclosure provides a process for ...,ZA2023/11713,"[{'id': '6167865', 'name': 'Toronto'}]",[],[NEW SCHOOL FOODS INC],13,84104569,N/A,...,0,2023,"[A23J3/24, A23V2002/00, A23J3/18, A23J3/06, A2...",<NA>,<NA>,0.726477,in,PB,[NEW SCHOOL FOODS INC],[New School Foods]
2,ZA-202306560-B,MEAT SUBSTITUTE COMPRISING ANIMAL MYOGLOBIN,Described herein is a meat substitute or food ...,ZA2023/06560,"[{'id': '2799397', 'name': 'Diest'}]",[],[PALEO B V],20,79927393,N/A,...,0,2023,"[A23J3/26, C07K14/805, A23L13/424, A23L33/17, ...",<NA>,<NA>,0.929412,in,F,[PALEO],[paleo]
7,ZA-202406920-B,FOOD PRODUCTS FROM ROOT VEGETABLES,A hard cheese analogue may be produced from a ...,ZA2024/06920,<NA>,[],[MCCAIN FOODS LTD],16,85704456,N/A,...,0,2024,"[A23L19/13, A23L19/09, A23L19/12, A23L19/15, A...",<NA>,<NA>,0.965656,in,PB,[MCCAIN FOODS LTD],[McCain Foods]
12,ZA-202401888-B,MICROBIAL-BASED PROCESS FOR IMPROVED QUALITY P...,The present invention describes a bio-based pr...,ZA2024/01888,"[{'id': '5226534', 'name': 'Brookings'}]",[],[PRAIRIE AQUATECH LLC],27,78845797,N/A,...,0,2024,"[A23J3/16, C12R2001/645, A23K20/147, Y02A40/81...",<NA>,<NA>,0.771990,in,F,[HOUDEK],[Houdek]
14,ZA-202400323-B,A METHOD FOR MANUFACTURING A FOOD PRODUCT FROM...,A method for manufacturing a food product from...,ZA2024/00323,<NA>,[],[USARIUM INC],14,82802741,N/A,...,0,2024,"[A23J3/16, A23J3/227, A23J3/26, A23J3/20, Y02P...",<NA>,<NA>,0.973452,in,F,[PLANETARIANS],[Planetarians]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,WO-2025129359-A1,EDIBLE SCAFFOLDS AND FORMULATIONS FOR CULTIVAT...,Cell culture scaffolds suitable for cultivated...,CA2024/051725,<NA>,[],[Meatleo Inc],1,96136087,Application,...,0,2024,"[C12N2533/78, C12N5/0068, C12N5/0658, A23L13/0...",[Meatleo Inc],"[{'doi': '10.1016/j.biomaterials.2022.121659',...",0.849341,in,CM,[MEATLEO INC],[Meatleo]
233,WO-2025127703-A1,ALTERNATIVE MEAT COMPRISING FUNGAL MYCELIUM AN...,The present specification relates to an altern...,KR2024/020267,<NA>,[],[Mynu Corp],3,96057899,Application,...,0,2024,"[A23L13/00, A23L31/00, A23K10/16, C12N1/14]",[Mynu Corp],<NA>,1.000000,in,F,[MYNU CORP],[Mynu Corp]
234,WO-2025126155-A1,COMPOSITIONS AND METHODS FOR PRODUCTION OF EDI...,Edible scaffolds prepared from plant protein a...,IB2024/062661,<NA>,[],[Umami Bioworks Pte Ltd],1,96056597,Application,...,0,2024,"[A23L13/00, A23J3/18, A23J3/20, A23J3/14, A23J...",[Umami Bioworks Pte Ltd],<NA>,0.947522,in,CM,[UMAMI BIOWORKS PTE LTD],[Umami Bioworks]
236,WO-2025120607-A1,CELL-CULTIVATED MILK FOR CONFECTIONERY AND DAI...,Cell-cultured milk generated in vitro in a rea...,IB2024/062345,<NA>,[],[Senara GmbH],1,94278617,Application,...,0,2024,"[A23C23/00, A23C9/20, C12N2510/00, C12M25/10, ...",[Senara GmbH],<NA>,0.810661,in,CM,[SENARA GMBH],[Senara]


In [81]:
df.head()

,id,title,abstract,application_number,assignee_cities,assignee_countries,assignee_names,family_count,family_id,filing_status,...,times_cited,year,cpc,current_assignee_names,publications,proba_scope,pred_combined,pred_pillar,assignee_names_cleaned,assignee_names_matched
0,ZA-202501977-B,A SOILLESS FARMING SYSTEM,The present invention relates to a soilless fa...,ZA2025/01977,"[{'id': '1259229', 'name': 'Pune'}]",[],[VISHWAKARMA INSTITUTE OF TECHNOLOGY],1,97493629,N/A,...,0,2025,<NA>,<NA>,<NA>,0.817097,in,PB,[VISHWAKARMA INSTITUTE OF TECHNOLOGY],[VISHWAKARMA INSTITUTE OF TECHNOLOGY]
1,ZA-202311713-B,"PROCESS FOR PRODUCING COOKABLE, FIBROUS MEAT A...",The present disclosure provides a process for ...,ZA2023/11713,"[{'id': '6167865', 'name': 'Toronto'}]",[],[NEW SCHOOL FOODS INC],13,84104569,N/A,...,0,2023,"[A23J3/24, A23V2002/00, A23J3/18, A23J3/06, A2...",<NA>,<NA>,0.726477,in,PB,[NEW SCHOOL FOODS INC],[New School Foods]
2,ZA-202306560-B,MEAT SUBSTITUTE COMPRISING ANIMAL MYOGLOBIN,Described herein is a meat substitute or food ...,ZA2023/06560,"[{'id': '2799397', 'name': 'Diest'}]",[],[PALEO B V],20,79927393,N/A,...,0,2023,"[A23J3/26, C07K14/805, A23L13/424, A23L33/17, ...",<NA>,<NA>,0.929412,in,F,[PALEO],[paleo]
3,ZA-202500303-B,AN ISOLATION AND CULTURE METHOD AND APPLICATIO...,The present invention belongs to the field of ...,ZA2025/00303,"[{'id': '1797929', 'name': 'Qingdao'}]",[],[UNIV QINGDAO],3,93807478,N/A,...,0,2025,"[C12N5/0653, C12N5/0668, C12N5/0654, C12N5/068...",<NA>,<NA>,0.722953,in,CM,[UNIV QINGDAO],[UNIV QINGDAO]
4,ZA-202500087-B,RESEARCH METHOD FOR SKM-EXOS THAT PROMOTE PROL...,The invention discloses a research method for ...,ZA2025/00087,"[{'id': '1814906', 'name': 'Chongqing'}]",[],[UNIV SOUTHWEST],2,91602588,N/A,...,0,2025,"[G01N33/5061, G01N21/6458, C12Q1/6876, G01N33/...",<NA>,<NA>,0.640767,in,CM,[UNIV SOUTHWEST],[UNIV SOUTHWEST]


In [57]:
## Check labels
df_training = pd.read_csv("../../Patents/Patents_Data/patents_subest_for_pipeline_testing2.csv")
df_training["subpillarT"] = df_training["subpillar"]
df_training["research_categoryT"] = df_training["research_category"]
df_training["endproductT"] = df_training["endproduct"]
df_training["ingredientT"] = df_training["ingredient"]

In [58]:
df_training = df_training[["id", "scope", "pillar", "subpillarT", "research_categoryT", "endproductT", "ingredientT"]]

In [22]:
len(df_training)

820

In [59]:
df_comb = pd.merge(df, df_training, on = "id")

In [53]:
len(df_comb)

525

In [60]:
df_comb

,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,date_dimensions,date_ML,...,research_category,end_product,ingredient,subpillar,scope,pillar,subpillarT,research_categoryT,endproductT,ingredientT
0,KR-20260058202-A,87847814,KR1020267005189,Powdered food emulsion composition containing ...,The present invention relates to a powdered fo...,"['A23P10/40', 'A23L2/39', 'A23C11/06', 'A23C11...",2026,KR,260710,260710,...,End product formulation,Cross-cutting,"Emulsions, gels, and binders",NaN,in,PB,NaN,Ingredient optimisation,Milk and milk proteins,"Emulsions, gels, and binders"
1,WO-2025074118-A1,93119608,GB2024/052562,METHODS FOR INCREASING THE FAT CONTENT OF A CELL,Described herein are novel methods for increas...,"['C12N5/0653', 'C12N2501/01', 'C12N2527/00']",2025,WO,260710,260710,...,Cell line development,Meat,Fats and oils,NaN,in,CM,NaN,Cell line development,Meat,Fats and oils
2,EP-4638706-A2,89542141,EP23838012.5,NANOFIBROUS SCAFFOLD FOR CELL CULTURING,The present invention relates to a method for ...,"['C12N2513/00', 'C12M33/00', 'C12M25/02', 'C12...",2025,EP,260710,260710,...,Scaffolding,Agnostic,N/A,NaN,in,CM,NaN,Scaffolding,Meat,NaN
3,PL-247657-B1,91227479,PL442891,Method for producing instant vegetable steaks ...,The subject of the application is an instant v...,"['A23L19/00', 'A23L33/125', 'A23L33/105', 'A23...",2025,PL,260710,260710,...,End product formulation,Meat,N/A,NaN,in,PB,NaN,End product formulation,Meat,NaN
4,US-20250000129-A1,83692816,US18690539,USE OF LEGUMINOUS STARCH AND ITS CROSS LINKED ...,The present invention is relative to the use o...,"['A23L29/219', 'A23L13/426', 'A23J3/227', 'A23...",2025,US,260710,260710,...,End product formulation,Meat,"Emulsions, gels, and binders",NaN,in,PB,NaN,Texturization methods,Meat,"Emulsions, gels, and binders"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
520,US-20210309723-A1,70469698,US17287905,RECOMBINANT PRODUCTION OF A COLLAGEN PEPTIDE P...,The present invention relates to a method for ...,"['A61K38/39', 'A61K38/00', 'A61Q19/00', 'A61Q5...",2021,US,260710,260710,...,Target molecule selection,Agnostic,"Emulsions, gels, and binders",PF,in,F,NaN,Target molecule selection,Cross-cutting,"Emulsions, gels, and binders"
521,EP-4750886-A1,87517248,EP24752137.0,METHOD OF GENERATING ADIPOCYTES,The invention relates to methods of generating...,"['C07K14/4702', 'C12N2501/115', 'G01N33/5044',...",2026,EP,260710,260710,...,Cell line development,Agnostic,N/A,NA,in,CM,NaN,Cell line development,Meat,Fats and oils
522,JP-2026504736-A,85035928,JP2025536188,Culture method,The present invention relates to a method for ...,"['C12N5/0062', 'C12N5/0075', 'C12N2509/00', 'C...",2026,JP,260710,260710,...,Bioprocess design,Agnostic,N/A,NA,in,CM,NaN,Bioprocess design,Meat,NaN
523,CN-118434289-A,79316900,CN202280083734.3,Method for improving processing characteristic...,The present invention relates to protein compo...,"['C12N9/63', 'A61K9/14', 'C08J2389/00', 'A23K2...",2024,CN,260710,260710,...,Ingredient optimisation,Cross-cutting,"Isolates, concentrates, and flours",NA,in,CC,NaN,Ingredient optimisation,Agnostic,"Isolates, concentrates, and flours"


In [61]:
df_comb["categoryC"] = df_comb["research_category"] == df_comb["research_categoryT"]
df_comb["endproductC"] = df_comb["end_product"] == df_comb["endproductT"]
df_comb["ingredientC"] = df_comb["ingredient"] == df_comb["ingredientT"]
df_comb["subpillarC"] = df_comb["subpillar"] == df_comb["subpillarT"]

In [41]:
df_comb.to_csv("../../Patents/Patents_Data/Training_labels_results.csv")

In [67]:
db.close()
